In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transform
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transforms = transform.Compose([
  transform.Resize((224, 244)),
  transform.ToTensor()
])

train_dataset = datasets.ImageFolder("cats_and_dogs_filtered/train", transform = transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

class CatDogCNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv = nn.Sequential(
      nn.Conv2d(3, 32, 3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(2),

      nn.Conv2d(32, 64, 3, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(2)
    )
    self.fc = nn.Sequential(
      nn.Flatten(),
      nn.Linear(64 * 56 * 56, 128),
      nn.ReLU(),
      nn.Linear(128, 1)
    )

  def forward(self, x):
    x = self.conv(x)
    return self.fc(x)
  
model = CatDogCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in range(5):
  total_loss = []
  for images, labels in train_loader:
    output = model(images)
    loss = criterion(output, labels)
    reg = 0
    reg += torch.sum([p ** 2 for p in model.parameters()])
    loss += reg * 1e-4
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss.append(loss)

In [ ]:
class CustomDropout(nn.Module):
  def __init__(self, p=0.5):
    super().__init__()
    self.p = 0.5

  def forward(self, x):
    if self.training:
      mask = torch.bernoulli(torch.ones_like(x), (1-self.p))
      return (x * mask) / (1-self.p)
    else:
      return x

In [ ]:
best_loss = float('inf')
counter = 0
patience = 2

for epoch in range(5):
  total_loss = []
  for x, y in train_loader:
    y_p = model(x)
    loss = criterion(y_p, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss.append(loss)
  if total_loss[-1] < best_loss:
    best_loss = total_loss[-1]
    counter = 0
  else:
    counter += 1
  if counter > patience:
    break

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

df = pd.read_csv('daily.csv').dropna()
y = df['Price'].values

sequence_len = 10

X, Y = [], []
for i in range(len(y) - sequence_len - 1):
  X.append(y[i:i+sequence_len])
  Y.append(y[i+sequence_len])

X_train = X[:int(0.8*len(X))]
Y_train = Y[:int(0.8*len(Y))]
X_test = X[int(0.8*len(X)):]
Y_test = Y[int(0.8*len(Y)):]

class NGTimeSeries(Dataset):
  def __init__(self, x, y):
    super().__init__()
    self.x = torch.tensor(x, dtype=torch.float32)
    self.y = torch.tensor(y, dtype=torch.float32)

  def __len__(self):
    return len(self.x)
  
  def __getitem__(self, index):
    return self.x[index], self.y[index]

train_dataset = NGTimeSeries(X_train, Y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = NGTimeSeries(X_train, Y_train)
test_loader = DataLoader(test_dataset, batch_size=32)

class RNNModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.rnn = nn.RNN(input_size=1, hidden_size=5, batch_first=True)
    self.fc = nn.Linear(5, 1)
  
  def forward(self, x):
    out, _ = self.rnn(x)
    out = out[:, -1, :] # last time step
    return self.fc(out).squeeze()

rnn_model = RNNModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(rnn_model.parameters(), lr=0.001)

for epoch in range(100):
  for x, y in train_loader:
    x = x.view(-1, sequence_len, 1)
    optimizer.zero_grad()
    y_p = rnn_model(x)
    loss = criterion(y_p, y)
    optimizer.step()

with torch.no_grad():
  test_pred = rnn_model(test_dataset.x.view(-1, sequence_len, 1))

print(test_pred)

tensor([ 0.1327,  0.1096,  0.0848,  ..., -0.0196,  0.0081,  0.0164])


In [ ]:
import string
import os

all_letters = string.ascii_letters + ".,;'"
n_letters = len(all_letters)
data = './data/names'

names_map = {}
names_categories = []

for filename in os.listdir(data):
  category = filename.split('.')[0]
  names_categories.append(category)
  with open(os.path.join(data, category), encoding='utf-8') as f:
    names_map[category] = f.read().strip().split('\n')

class NameRNN(nn.modules):
  def __init__(self):
    self.rnn = nn.RNN(n_letters, hidden_size=128, batch_first=True)
    self.fc = nn.Linear(128, len(names_categories))
    self.softmax = nn.LogSoftmax(dim=1)

  def forward(self, x, hidden):
    x, hidden = self.rnn(x, hidden)
    x = self.fc(x[-1])
    return self.softmax(x), hidden

name_model = NameRNN()
criterion = nn.NLLLoss()

for i in range(1000):
  